# Run-Aggregate Statistik (llm-01)

Vergleicht die beiden neuesten `run_aggregate.json` der `state-presentation-llm-01` Runs.
Berechnet pro Run u.a. Mittelwert, Standardabweichung, Median, Min/Max, Spannweite
und Notenverteilung der `overall_score` Werte und stellt die beiden Runs gegenueber.

In [11]:
import json
import statistics as st
from collections import Counter
from pathlib import Path

SONNET_RUNS = [
    'run_2026-06-12_13-21-59_state-presentation-llm-01_sonnet4-5',
    'run_2026-06-15_07-55-43_state-presentation-llm-01_sonnet4-5',
    'run_2026-06-15_08-11-47_state-presentation-llm-01_sonnet4-5'
]

LLM_RUNS = ['run_2026-06-12_13-21-59_state-presentation-llm-01_sonnet4-5',
            'run_2026-06-12_13-26-51_state-presentation-llm-01_gpt4-5',
            'run_2026-06-15_07-38-53_state-presentation-llm-01_qwen3-5']

# Repo-Root relativ zum Notebook (playground/notebooks -> ../../)
REPO_ROOT = Path.cwd().parents[1]
RUNS_DIR = REPO_ROOT / "run_outputs"

for run_name in SONNET_RUNS:
    print(run_name)

run_2026-06-12_13-21-59_state-presentation-llm-01_sonnet4-5
run_2026-06-15_07-55-43_state-presentation-llm-01_sonnet4-5
run_2026-06-15_08-11-47_state-presentation-llm-01_sonnet4-5


In [ ]:
def load_aggregate(run) -> dict:
    # akzeptiert einen Run-Namen (str) ODER einen Path; loest Namen relativ zu RUNS_DIR auf
    run_dir = run if isinstance(run, Path) else RUNS_DIR / run
    return json.loads((run_dir / "run_aggregate.json").read_text())


def compute_stats(agg: dict) -> dict:
    files = agg["files"]
    scores = [f["overall_score"] for f in files.values()]
    pass_rates = [f["overall_pass_rate"] for f in files.values()]
    grades = Counter(f["quality_grade"] for f in files.values())
    return {
        "timestamp": agg["timestamp"],
        "n": len(scores),
        "mean": st.mean(scores),
        "stdev": st.stdev(scores),
        "pstdev": st.pstdev(scores),
        "median": st.median(scores),
        "min": min(scores),
        "max": max(scores),
        "range": max(scores) - min(scores),
        "cv": st.stdev(scores) / st.mean(scores),  # Variationskoeffizient
        "pass_rate_mean": st.mean(pass_rates),
        "pass_rate_stdev": st.stdev(pass_rates),
        "grades": dict(sorted(grades.items())),
        "scores": scores,
    }


stats = [compute_stats(load_aggregate(p)) for p in SONNET_RUNS]
for s in stats:
    print(f"\n=== {s['timestamp']}  (n={s['n']}) ===")
    print(f"  mean           : {s['mean']:.4f}")
    print(f"  stdev (sample) : {s['stdev']:.4f}")
    print(f"  pstdev (pop.)  : {s['pstdev']:.4f}")
    print(f"  median         : {s['median']:.4f}")
    print(f"  min / max      : {s['min']:.4f} / {s['max']:.4f}")
    print(f"  range          : {s['range']:.4f}")
    print(f"  CV             : {s['cv']:.4f}")
    print(f"  pass_rate mean : {s['pass_rate_mean']:.4f} (stdev {s['pass_rate_stdev']:.4f})")
    print(f"  grades         : {s['grades']}")

In [ ]:
# Direkter Vergleich der beiden Runs
older, newer = stats[0], stats[1]
metrics = ["mean", "stdev", "median", "min", "max", "range", "cv", "pass_rate_mean"]

print(f"{'Metrik':<16}{older['timestamp']:>22}{newer['timestamp']:>22}{'Delta':>12}")
print("-" * 72)
for m in metrics:
    delta = newer[m] - older[m]
    print(f"{m:<16}{older[m]:>22.4f}{newer[m]:>22.4f}{delta:>+12.4f}")

In [ ]:
# Paarweiser Vergleich der beiden Runs Datei fuer Datei.
# Misst, wie stark sich die Runs UNTERSCHEIDEN - nicht nur, ob die Mittelwerte gleich sind.
# (Zwei Runs koennen denselben Mittelwert haben und sich trotzdem bei jeder Datei stark unterscheiden.)

files_a = load_aggregate(last_two[0])["files"]
files_b = load_aggregate(last_two[1])["files"]
common = sorted(set(files_a) & set(files_b))

diffs = {f: files_b[f]["overall_score"] - files_a[f]["overall_score"] for f in common}
values = list(diffs.values())

mae = st.mean(abs(d) for d in values)            # mittlere absolute Abweichung
bias = st.mean(values)                            # mittlere vorzeichenbehaftete Differenz (Run B - Run A)
rmse = (st.mean(d ** 2 for d in values)) ** 0.5    # quadratisches Mittel

# Pearson-Korrelation der Scores zwischen den Runs
xa = [files_a[f]["overall_score"] for f in common]
xb = [files_b[f]["overall_score"] for f in common]
ma, mb = st.mean(xa), st.mean(xb)
cov = sum((x - ma) * (y - mb) for x, y in zip(xa, xb))
den = (sum((x - ma) ** 2 for x in xa) * sum((y - mb) ** 2 for y in xb)) ** 0.5
pearson = cov / den if den else float("nan")

print(f"Gemeinsame Dateien : {len(common)}")
print(f"MAE (mittl. abs.)  : {mae:.4f}   <- typische Abweichung pro Datei zwischen den Runs")
print(f"RMSE               : {rmse:.4f}")
print(f"Bias (B - A)       : {bias:+.4f}  <- ~0 => kein systematischer Trend")
print(f"Pearson r          : {pearson:.4f}   <- 1.0 = perfekt gleiche Rangordnung")
print(f"besser / schlechter / gleich : "
      f"{sum(d > 0 for d in values)} / {sum(d < 0 for d in values)} / {sum(d == 0 for d in values)}")

print("\nGroesste Einzel-Abweichungen:")
for f in sorted(common, key=lambda f: abs(diffs[f]), reverse=True)[:8]:
    print(f"  {f:<44} {files_a[f]['overall_score']:.3f} -> {files_b[f]['overall_score']:.3f}  ({diffs[f]:+.3f})")


In [ ]:
import re

# Varianz zwischen den Datensaetzen.
# Jeder Datensatz (z.B. geo_gdi-de) hat mehrere Wiederholungen (_02, _03).
# - within-Varianz : Streuung der Wiederholungen INNERHALB eines Datensatzes
# - between-Varianz: Streuung der Datensatz-Mittelwerte ZWISCHEN den Datensaetzen


def dataset_variance(agg: dict) -> dict:
    groups: dict[str, list[float]] = {}
    for fname, f in agg["files"].items():
        base = re.sub(r"(_\d+)?\.rdf$", "", fname)  # _02/_03-Suffix entfernen
        groups.setdefault(base, []).append(f["overall_score"])

    per_dataset = {
        base: {
            "n": len(sc),
            "mean": st.mean(sc),
            "variance": st.pvariance(sc) if len(sc) > 1 else 0.0,
            "stdev": st.pstdev(sc) if len(sc) > 1 else 0.0,
        }
        for base, sc in sorted(groups.items())
    }
    means = [d["mean"] for d in per_dataset.values()]
    within = [d["variance"] for d in per_dataset.values() if d["n"] > 1]
    return {
        "per_dataset": per_dataset,
        "n_datasets": len(per_dataset),
        "between_variance": st.pvariance(means),  # Varianz der Datensatz-Means
        "mean_within_variance": st.mean(within) if within else 0.0,
    }


for p, s in zip(last_two, stats):
    dv = dataset_variance(load_aggregate(p))
    print(f"\n=== {s['timestamp']}  ({dv['n_datasets']} Datensaetze) ===")
    for base, d in dv["per_dataset"].items():
        print(f"  {base:<38} mean={d['mean']:.4f}  within-var={d['variance']:.5f}  (σ={d['stdev']:.4f})")
    print(f"  -> between-dataset variance : {dv['between_variance']:.5f}")
    print(f"  -> mean within-dataset var  : {dv['mean_within_variance']:.5f}")


In [ ]:
# Optional: Verteilung der scores visualisieren
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
for s in stats:
    ax.hist(s["scores"], bins=10, alpha=0.5, label=f"{s['timestamp']} (\u03c3={s['stdev']:.3f})")
ax.set_xlabel("overall_score")
ax.set_ylabel("Anzahl Dateien")
ax.set_title("Verteilung overall_score \u2013 letzte 2 llm-01 Runs")
ax.legend()
plt.tight_layout()
plt.show()